# Project Title
### Data Engineering Capstone Project

#### Project Summary
An Analysis complany wants to analyse US Immigration data.Their main focus is on the immigrations done through 'Air' travelling. 

The project follows the follow steps:
* Step 1: Scope the Project and Gather Data
* Step 2: Explore and Assess the Data
* Step 3: Define the Data Model
* Step 4: Run ETL to Model the Data
* Step 5: Complete Project Write Up

In [ ]:
#import findspark
#findspark.init()
#import pyspark

In [ ]:
# Do all imports and installs here
import pandas as pd
import configparser
import os
import pandas as pd
from datetime import datetime
from pyspark.sql import SparkSession
from pyspark.sql.functions import udf,col,explode,monotonically_increasing_id
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, TimestampType, DateType

### Step 1: Scope the Project and Gather Data

#### Scope 
In this project I am going to build a data model that connects the given dataset and explore the data belongs United States and the immigration through 'Air' mode.
#### Datasets Used
The following datasets are included in the project.
* **I94 Immigration Data:** This data comes from the US National Tourism and Trade Office. A data dictionary is included for working by Udacity. [This](https://travel.trade.gov/research/reports/i94/historical/2016.html) is where the data comes from. 
    * _`i94_apr16_sub.sas7bdat`_ is the original one of the files with complete data present on  Udacity disk. 
    * `'../../data/18-83510-I94-Data-2016/i94_apr16_sub.sas7bdat'` path in Udacity workspace for the file which has lot of data processing it takes lot of time on local system.
    * `I94_SAS_Labels_Descriptions.SAS` is the Label Desctiptor file for the SAS file which has valid and invalid codes present in the I94 Immigration Data file.
    * There's a sample file so you can take a look at the data in csv format before reading it all in. `immigration_data_sample.csv`. 
* **World Temperature Data:** This dataset came from Kaggle. You can read more about it [here](https://www.kaggle.com/berkeleyearth/climate-change-earth-surface-temperature-data).
    *  `../../data2/GlobalLandTemperaturesByCity.csv` is path for the file on Udacity workspace
* **U.S. City Demographic Data:** This data comes from OpenSoft. You can read more about it [here](https://public.opendatasoft.com/explore/dataset/us-cities-demographics/export/). - `us-cities-demographics.csv`
* **Airport Code Table:** This is a simple table of airport codes and corresponding cities. It comes from [here](https://datahub.io/core/airport-codes#data). - `airport-codes_csv.csv`

_NOTE:_ 
    1. All the `csv` data Sample Files are present in the `Datasets` Folder
    2. Ignore the extra `immigration_data_sample.csv` has an extra column

In [ ]:
### Follwing is the code for working in Udacity Workspace
	
from pyspark.sql import SparkSession
spark = SparkSession.builder.\
config("spark.jars.packages","saurfang:spark-sas7bdat:2.0.0-s_2.11")\
.enableHiveSupport().getOrCreate()

df_spark =spark.read.format('com.github.saurfang.sas.spark').load('../../data/18-83510-I94-Data-2016/i94_apr16_sub.sas7bdat')

#write SAS data to parquet
df_spark.write.mode("overwrite").partitionBy("i94yr","i94mon").parquet("Datasets_created_by_Scripts/sas_data")
immigration_df=spark.read.parquet("Datasets_created_by_Scripts/sas_data")
#df_spark.printSchema()

In [ ]:
###################################################################################
### Read in 'us-cities-demographics.csv' and  'airport-codes_csv.csv' data here ###
###################################################################################

us_cities_df = spark.read.csv("Datasets/us-cities-demographics.csv",sep = ";",header = True)
airport_codes_df = spark.read.csv("Datasets/airport-codes_csv.csv",header = True)

#df = spark.read.csv("Datasets/immigration_data_sample.csv")

#us_cities_df.printSchema()
#airport_codes_df.printSchema()

In [ ]:
#########################################################################################################
### 1. Read in 'GlobalLandTemperaturesByCity.csv' data.                                               ###
### 2. As the date is huge writing partitiong is by country and city and writing into parquet format  ###
#########################################################################################################

global_temp_df = spark.read.csv('../../data2/GlobalLandTemperaturesByCity.csv',header=True)
global_temp_df.write.mode("overwrite").partitionBy("country","city").parquet("Datasets_created_by_Scripts/global_temperatures")
global_tempetature_df = spark.read.parquet("Datasets_created_by_Scripts/global_temperatures")

#global_tempetature_df.printSchema()

In [ ]:
airport_codes_df.show(3)
us_cities_df.show(3)
global_tempetature_df.show(3)

### Gathering Data
In the above immigration data there are 28 columns out of which 7 are *I94* columns.
  1. **I94YR**   - Year
  2. **I94MON**  - Month 
  3. **I94CIT**  - City Codes
  4. **I94RES**  - Residence Codes
  5. **I94PORT** - PORT Codes
  6. **I94MODE** - Mode of Immigration Codes
  7. **I94VISA** - VISA Codes

Other than I94YR and I94MON all other I94 columns are populated with respective codes. The descriptions for these codes are present in  _`I94_SAS_Label_Descriptors.SAS`_ file.

##### Labels and Codes from _I94_SAS_Label_Descriptors.SAS_ file
Taking the _`I94_SAS_Label_Descriptors.SAS`_ I have  created a  **nested** json file - `I94_SAS_Label_Descriptors.json`. <br> 
File coversion is done using  `class LabelDescriptorToJson` from **`LabelDescriptorToJson.py`** script.
      
      class takes 3 arguments as inputs:
          input_file_path - path to `I94_SAS_Label_Descriptors.SAS` including filename
          labels - all the `I94` columns names that are coded
          output_file_path - output json file path with filename

Invoking class : **`LabelDescriptorToJson(input_file,labels,output_file)`**

Following is the JSON file structure created by script
           
   ```json
   [
    {
      "I94CIT_I94RES": [{"key": "582", "val": "MEXICO Air Sea, and Not Reported (I-94, no land arrivals)"}, 
                    {"key": "236", "val": "AFGHANISTAN"}, {"key": "101", "val": "ALBANIA"},......]
    },
    {
      "I94PORT": [{"key": "ALC", "val": "ALCAN, AK"}, {"key": "ANC", "val": "ANCHORAGE, AK"},... ]
    },
    {
      "I94MODE": [{"key": "1", "val": "Air"}, {"key": "2", "val": "Sea"}, {"key": "3", "val": "Land"}, {"key": "9", "val": "Not reported"}]
    }, 
    {
      "I94ADDR": [{"key": "AL", "val": "ALABAMA"}, {"key": "AK", "val": "ALASKA"}, {"key": "AZ", "val": "ARIZONA"}, ...]
    }, 
    {
      "I94VISA": [{"key": "1", "val": "Business"}, {"key": "2", "val": "Pleasure"}, {"key": "3", "val": "Student"}]
    }
]
```

In [ ]:
####
#### Importing LabelDescriptorToJson class and creating JSON file from the I94_SAS_Labels_Descriptions.SAS file
####
from LabelDescriptorToJson import LabelDescriptorToJson
inpt_file = 'Datasets/I94_SAS_Labels_Descriptions.SAS'
labels = ['I94CIT', 'I94RES','I94PORT','I94MODE','I94ADDR','I94VISA']
opt_file = 'Datasets_created_by_Scripts/I94_SAS_Label_Descriptors.json'
LabelDescriptorToJson(inpt_file,labels,opt_file)

### Step 2: Explore and Assess the Data
#### Explore the Data 
Identifying data quality issues, like missing values, duplicate data, etc. <br>

Reading the JSON data created in the above steps and exploding the I94 columns seperately to find the relations with other datasets -> us-cities-demographics.csv, airport-codes_csv.csv, GlobalLandTemperaturesByCity.csv 

#### Cleaning Steps
1. Selecting only the required columns from the immigration table.
2. Filtering the data by 'Air' mode i.e., i94mode = 1. Following are three modes of immigrations to US 
    * 1 = 'Air'
    * 2 = 'Sea'
    * 3 = 'Land'
    * 9 = 'Not reported'
   
3. Selecting necessary columns from Datasets/us-cities-demographics.csv
4. Since we are dealing with only US immigration data filtering the `/data/data2/GlobalLandTemperaturesByCity.csv` by `country = 'United States'`
5. Filtering the global tempetatures dataset by removing `null` temperature values i.e., AverageTemperature AND AverageTemperatureUncertainty with null values


In [ ]:
####################################################################################
####################################################################################
####                                                                            ####
#### Reading the json data and exploding each and every  I94 columns seperetely ####
####                                                                            #### 
####################################################################################
####################################################################################
label_desc_df = spark.read.json("Datasets_created_by_Scripts/I94_SAS_Label_Descriptors.json")


modes_df =label_desc_df.select(explode(col("I94MODE"))).toDF("mode").selectExpr("CAST(mode.key AS INT) mode_id","mode.val as mode_name") # MODES DIMENSION TABLE
addr_df = label_desc_df.select(explode(col("I94ADDR"))).toDF("addr").selectExpr("addr.key as state_code","addr.val as state")  # STATES TABLE
city_res_df = label_desc_df.select(explode(col("I94CIT_I94RES"))).toDF("city_res").selectExpr("CAST(city_res.key AS INT) city_code","city_res.val as city") # CITIES DIMENSION TABLE
ports_df = label_desc_df.select(explode(col("I94PORT"))).toDF("port").selectExpr("port.key as port_code","port.val as port_name") # PORTS TABLE
visa_code_df = label_desc_df.select(explode(col("I94VISA"))).toDF("visa").selectExpr("CAST(visa.key AS INT) visa_id","visa.val as visa_name") # VISA DIMENSON TABLE


In [ ]:
###
### Following sample datasets and schema for thr SAS Descriptors which will be loaded into 4 Dimension tables
###

#modes_df.show()      # MODES DIMENSION TABLE
#addr_df.show(3)      # STATES DIMENSION TABLE
#city_res_df.show(3)  # CITIES DIMENSION TABLE
#ports_df.show(3)     # PORTS DIMENSION TABLE
#visa_code_df.show(3) # VISA DIMENSON TABLE


modes_df.printSchema()      # MODES DIMENSION TABLE
addr_df.printSchema()      # STATES DIMENSION TABLE
city_res_df.printSchema()  # CITIES DIMENSION TABLE
ports_df.printSchema()     # PORTS DIMENSION TABLE
visa_code_df.printSchema() # VISA DIMENSON TABLE

In [ ]:
###
### Following are some of the invalid Codes present in the immigration data
###
#city_res_df.select(['city_res_code','value']).where("value like '%INVALID%'").show()
#airport_codes_df.select(['airport_code','value']).where("value like '%INVALID%'").show()
#airport_codes_df.select(['airport_code','value']).where("value like 'No PORT Code%'").show()

In [ ]:
# Performing cleaning tasks here
############################################################################################
############################################################################################
###                                                                                      ###
### Exploring Immigration data read from 'i94_apr16_sub.sas7bdat'  into 'immigration_df' ###
###                                                                                      ###
############################################################################################
############################################################################################

# already read before
# immigration_df=spark.read.parquet("Datasets_created_by_Scripts/sas_data")

###
### Filtering the data with immigration mode of Air
###
immigration_df = immigration_df.filter(col('i94mode')== 1)
immigration_df.show(5)

In [ ]:
#############################################################################################################################
#############################################################################################################################
######## udf definition to convert the sas date numeric to DateType()                                                ########
########                                                                                                             ########
######## SAS date value is a value that represents the number of days between January 1, 1960, and a specified date. ########
######## Therefore, if we convert the numbers to Pandas Timedeltas and add them to 1960-1-1 we can recover the date  ########
#############################################################################################################################
#############################################################################################################################

### convert_sas_date = udf(lambda epoch: pd.to_timedelta(epoch, unit='D') + pd.datetime(1960, 1, 1),DateType())

@udf(DateType())
def convert_sas_date(epoch):
    if epoch:
        time = pd.to_timedelta(epoch, unit='D') + pd.datetime(1960, 1, 1)
        return time
    else:
        return datetime(9999,9,9)
    


In [ ]:
###
### Selecting only the required columns
###
df = immigration_df.selectExpr(["CAST(cicid AS BIGINT) ",
                                    "CAST(i94yr AS INT) ",
                                    "CAST(i94mon AS INT) ",
                                    "CAST(i94cit AS INT)  ",
                                    "CAST(i94res AS INT) ",
                                    "i94port",
                                    "CAST(arrdate AS BIGINT) arrival_date ",
                                    "CAST(i94mode as INT) ",
                                    "i94addr",
                                    "CAST(depdate AS BIGINT) departure_date ",
                                    "CAST(i94visa AS INT)",
                                    "CAST(biryear AS INT) ",
                                    "gender",
                                    "visatype"])


###
### following .withColumn will overwrite the sas date numberic with dateType() as the given alias column name is same as the inputcolumn name i.e., arrival_date and departure_date
###
immig_df = df.withColumn("arrival_date",convert_sas_date("arrival_date")).withColumn("departure_date",convert_sas_date("departure_date"))

In [ ]:
###
### Checking for any null columns
###

immig_df.filter("i94cit IS NULL or i94res IS NULL  or i94mode IS NULL  or i94visa IS NULL or biryear IS NULL").show(3)
immig_df.filter("i94port IS NULL or i94addr IS NULL or gender IS NULL").show(3)
immig_df.filter("visatype IS NULL").show()

In [ ]:
###
### Replacing Null columns with Default values
###
col_defaults = {
                 "gender" :'NA',
                 "i94addr":'NO ADDR',
                 "biryear" : -999999
}

cleansed_immig_df = immig_df.na.fill(col_defaults)
clean_immig_df.show()


In [ ]:
###
### DF of exploded 'I94addr' from SAS Json file 
###

addr_df.show(5)  # STATES DIMENSION TABLE

#us_cities_df = spark.read.csv("Datasets/us-cities-demographics.csv",sep = ";",header = True) #This DF was run in Step 1 so no need to run again

#################################################################
#################################################################
####                                                         ####
#### Exploring and cleaning  us-cities-demographics.csv data ####
####                                                         ####
#################################################################
#################################################################

us_cities_df.show(5)

In [ ]:
###
### From the above I94 Codes we can relate 'addr_codes' with 'state_codes' in demographics
###
df = us_cities_df.selectExpr(["City as city",
                                          "State as state",
                                          "CAST(`Median Age` AS DOUBLE) median_age",
                                          "CAST(`Male Population` AS INT) male_population",
                                          "CAST(`Female Population` AS INT) female_population",
                                          "CAST(`Total Population` AS INT) total_population",
                                         "CAST(`Number of Veterans` AS INT) as veterans_count",
                                         "CAST(`Foreign-born` AS INT) foriegn_born",
                                          "Race",
                                          "`State code` AS state_code"
                                         ])

#cleansed_us_cities_df.show(5)

####
#### US_CITY_DEMOGRAPHICS TABLE
####
#### us-cities-demographics can be connected by 'state_code' with 'STATE_CODE' column STATES DIMENSION TABLE
####

In [ ]:
###
### Setting default values for the null columns
###
df.filter("foriegn_born IS NULL").show(5)
cleansed_us_cities_df = df.na.fill(-999,["veterans_count","foriegn_born"])
cleansed_us_cities_df.filter("foriegn_born IS NULL").show(5)
cleansed_us_cities_df.filter("foriegn_born = -999").show(5)

In [ ]:
# already read before
#global_tempetature_df = spark.read.parquet("Datasets_created_by_Scripts/global_temperatures")

########################################################################
########################################################################
#####                                                              #####
##### Exploring and cleaning GlobalLandTemperaturesByCity.csv data #####
#####                                                              #####
########################################################################
########################################################################

global_tempetature_df.show(5)

In [ ]:
#United States
###
### Creating temporary view for global_tempetature_df
###
global_tempetature_df.createOrReplaceTempView("globalTempTable")
cleansed_us_cities_df.createOrReplaceTempView("us_demographics")

In [ ]:
####
#### Checking for the Null temperature records
####
spark.sql("""SELECT count(1) US_TemperaturesCount 
                    FROM globalTempTable 
                    WHERE Country = 'United States'""").show()

spark.sql("""SELECT count(1) US_TemperaturesWithNull_records
                    FROM globalTempTable 
                    WHERE AverageTemperature IS NULL AND AverageTemperatureUncertainty IS NULL 
                            AND Country = 'United States'""").show()

In [ ]:
####
#### Cleaning the GlobalLandTemperaturesByCity.csv data and selecting only 'United States' records
####

cleansed_global_temp_df = spark.sql("""SELECT TO_DATE(dt) as noted_date,
                                                CAST(AverageTemperature AS DOUBLE) Average_Temperature,
                                                CAST(AverageTemperatureUncertainty AS DOUBLE) Average_Temperature_Uncertainty,   
                                                City,      
                                                Country,
                                                Latitude,
                                                Longitude FROM globalTempTable 
                                                WHERE AverageTemperature IS NOT NULL 
                                                AND AverageTemperatureUncertainty IS NOT NULL  
                                                AND Country = 'United States'""")
cleansed_global_temp_df.show(5)

#####
##### TEMPERATURES TABLE
#####
##### CITY column of globalTempperature data can be joined CITY of US_CITY_DEMOGRAPHICS table
#####
cleansed_global_temp_df.printSchema()

In [ ]:
# already read before
# airport_codes_df = spark.read.csv("Datasets/airport-codes_csv.csv",header = True)

###################################################
###################################################
#####                                         #####
##### Exploring airport-codes_csv.csv dataset #####
#####                                         #####
###################################################
###################################################

###
### AIRPORTS TABLE
### airport_codes can be connected by 'local_code' with 'PORT_CODE' column PORTS DIMENSION TABLE
###


clean_airport_codes_df = airport_codes_df.filter(col("iso_country")== 'US')

clean_airport_codes_df = clean_airport_codes_df.selectExpr("ident",
                                                "type",
                                                "name",
                                                "CAST(elevation_ft as INT)",
                                                "iso_country",
                                                "iso_region",
                                                "municipality",
                                                "gps_code",
                                                "local_code",
                                                "coordinates")
clean_airport_codes_df.printSchema() 



### Step 3: Define the Data Model
#### 3.1 Conceptual Data Model
Following is the Conceptual model  that can be formed by exploring the above datasets

<img src = "./images/Immigration_CONCEPTUAL_Schema.jpg">

#### 3.2 Mapping Out Data Pipelines
To load the data into above modelled table following are the steps:
1. As the data is huge first load the data files on to S3 bucket
2. Processing in them using spark would be easy considering is Schema on read property and the size of the data we are dealing with.
3. Processed data are to be loaded into Redshift cluster using postgres jdbc connection

In [ ]:
###
### CREATING TEMPORARY VIEWS FOR RUNNING SOME QUERIES
###
airport_codes_df.createOrReplaceTempView("airport_codes_table")
ports_df.createOrReplaceTempView("ports_table")
cleansed_immig_df.createOrReplaceTempView("immigration_table")
addr_df.createOrReplaceTempView("states")

### Step 4: Run Pipelines to Model the Data 
#### 4.1 Create the data model
All the above code snippets in the Jupyter notebook are used in the following scripts.
Following are the scripts to build the above mentioned piepline
   1. LoadingDataFilesToS3.py - Before loading on to S3, `LabelDescriptorToJson` class from LabelDescriptorToJson.py is called to convert the SAS labels into JSON file
   2. CreateTables.py - Creates Tables on Redshift - Uses sql_queries.py which contains CREATE and DROP statements
   3. ETL2Redshift.py - Processes the data and load into Redshift

In [ ]:
# This script shoud be run on EMR
%run LoadingDataFilesToS3.py

In [ ]:
%run CreateTables.py

In [ ]:
# This should be run on EMR cluster as the process takes too long
%run ETL2Redshift.py

#### 4.2 Data Quality Checks
Explain the data quality checks you'll perform to ensure the pipeline ran as expected. These could include:
 * Integrity constraints on the relational database (e.g., unique key, data type, etc.)
 * Unit tests for the scripts to ensure they are doing the right thing
 * Source/Count checks to ensure completeness
 
Run Quality Checks

In [ ]:
# Perform quality checks here
#connects to you  Redshift Cluster
import configparser
config = configparser.ConfigParser()
config.read_file(open('cloud.cfg'))
(HOST,DB_NAME, DB_USER, DB_PASSWORD, DB_PORT) = config['CLUSTER'].values()

conn_string="postgresql://{}:{}@{}:{}/{}".format(DB_USER, DB_PASSWORD, HOST, DB_PORT,DB_NAME)
print(conn_string)

In [ ]:
%load_ext sql

In [ ]:
%%sql 
SELECT i94mode,COUNT(*) FROM IMMIGRANTS WHERE i94mode=1;

In [ ]:
%%sql
SELECT * from visas;

In [ ]:
%%sql
SELECT * from US_CITY_DEMOGRAPHICS;

In [ ]:
%%sql
Select count(*) from AIRPORTS;

In [ ]:
%%sql
SELECT p.port_name,name,
        continent,
        iso_country,
        iso_region,
        municipality,
        local_code,
        coordinates FROM AIRPORTS a 
        JOIN PORTS p ON a.local_code = p.port_code  
WHERE iso_country = 'US' LIMIT 5;

#### 4.3 Data dictionary 
Th data Disctionary is provired in a seperate file - `DataDictionary`

#### Step 5: Complete Project Write Up

* I have used Apacke Spark to read, Transform and Load the data.While working with huge volumes of data spark processing speed is high and data manipulation and cleansing though dtataframes is very easy.
* Data of the immigration departments will be high. To get the recent data for analysis updating it once in a Month would be prefereble, so that analysis for how much inflow to the country is increased or decreased an be kept track.
* If the data is increased by 100x I would run my scripts though a cluster though using Amazon EMR. an dLoad it into Redshift so that it can be accessed by 100+ people
* Automating cexecution order of the scripts can be done though scheduling tools. Assuming this huge data to be processed, ETL Job can be scheduled daily at 7pm so that data is updated by 7am every day.